# Option Quotes 样本内层 ZIP 字段核查

只读已归档原始 ZIP，核对五个嵌套 CSV 的表头和前两行。不下载、不拟合、不从通用样本推断 SOXX 历史覆盖。

In [1]:
import json,zipfile,io,hashlib
from pathlib import Path
root=Path.cwd();card=root/'research/soxx_option_vendor_sample_preflight_20261023';raw=root/'data/snapshots/soxx_option_vendor_sample_preflight_20261023';sha=lambda f:hashlib.sha256(f.read_bytes()).hexdigest()
manifest=json.loads((card/'sample_manifest.json').read_text());assert sha(raw/'option_quotes.zip')==next(x['sha256'] for x in manifest['files'] if x['name']=='option_quotes.zip')
rows=[]
with zipfile.ZipFile(raw/'option_quotes.zip') as outer:
 for member in outer.infolist():
  if not member.filename.lower().endswith('.zip'):continue
  with zipfile.ZipFile(io.BytesIO(outer.read(member))) as inner:
   files=[x for x in inner.infolist() if not x.is_dir()]
   assert len(files)==1 and files[0].filename.lower().endswith('.csv')
   with inner.open(files[0]) as f:lines=[next(f).decode('utf-8-sig','replace').strip() for _ in range(3)]
   rows.append({'outer_member':member.filename,'inner_member':files[0].filename,'inner_bytes':files[0].file_size,'csv_header':lines[0],'first_two_rows':lines[1:]})
assert len(rows)==5
result={'status':'five_nested_Option_Quotes_sample_CSV_schemas_inspected_no_SOXX_coverage_claim','source_ZIP':'option_quotes.zip','nested_CSVs':rows,'new_external_GETs':0,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'nested_profiles.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
print({'nested_CSVs':len(rows),'sample_underlyings':[x['first_two_rows'][0].split(',')[0] for x in rows]})

{'nested_CSVs': 5, 'sample_underlyings': ['AAPL', 'AAPL', 'AAPL', 'AAPL', '^SPX']}


## 中文结论

五个内层 CSV 样本首行仅见 AAPL 或 ^SPX；样本有 quote_datetime、bid/ask，含 Calcs 版本有 IV/Delta。仍需 SOXX 专属历史样本与发布时间证据。